# Multivariate Regression

Symbolic regression on a higher-dimensional dataset with feature selection.
The target is a multivariate function combining several input features,
demonstrating DRAGON's ability to handle many candidate features.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

notebook_dir = Path().resolve()
repo_root = notebook_dir.parents[2] if notebook_dir.name == "SymbolicRegression" else notebook_dir
lib_dir = repo_root / "lib"
if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## 1. Generate Multivariate Data

Target: $y = x_1^2 + 2 x_2 x_3 - x_4 / x_5$ with 10 input features
(only 5 are used in the true formula; the rest are distractors).

In [ ]:
n_samples = 3000
n_features = 10
rng = np.random.default_rng(42)

X_np = rng.uniform(-5, 5, (n_samples, n_features))
y_np = X_np[:, 0]**2 + 2 * X_np[:, 1] * X_np[:, 2] - X_np[:, 3] / (np.abs(X_np[:, 4]) + 0.1)

col_names = [f"x{i}" for i in range(n_features)]
X = pd.DataFrame(X_np, columns=col_names)
y = pd.Series(y_np, name="y")

print(f"X shape: {X.shape},  y range: [{y.min():.2f}, {y.max():.2f}]")
print("True formula: y = x0^2 + 2*x1*x2 - x3/|x4|")
X.head()

## 2. Feature Selection

Select the top features from 10 candidates using gradient boosting importance.

In [ ]:
from dragon.utils.symbolic.features_selection import GradientBoostingSelector

selector = GradientBoostingSelector(n_top=5, model_type="xgboost")
top_features, scores = selector.fit_transform(X, y)
num_features = len(top_features)

print("Top features:", top_features)
print("Scores:", {k: round(v, 4) for k, v in scores.items()})

## 3. Build the Search Space

In [ ]:
from dragon.search_space.bricks import (
    Identity, SelectFeatures, Inverse, Negate, Power,
    SumFeatures, ConstantBrick, Ln, Sin, Cos, Exp, ExpAffine, ChannelBoost,
)
from dragon.search_space.bricks_variables import operations_var
from dragon.search_space.base_variables import CatVar, Constant, ArrayVar
from dragon.search_space.dag_encoding import SymbolicNode, AdjMatrix
from dragon.search_space.dag_variables import HpVar, EvoDagVariable
from dragon.search_operators.base_neighborhoods import (
    CatInterval, ConstantInterval, ArrayInterval,
)
from dragon.search_operators.dag_neighborhoods import EvoDagInterval, HpInterval

MAX_NODES = 15

from dragon.utils.symbolic.features_selection import CombinationBuilder
combo_builder = CombinationBuilder()
all_combos = combo_builder.build(top_features, scores)
combo_weights = SelectFeatures.combination_weights(
    [scores.get(c, 1.0 / num_features) for c in top_features], all_combos)

def hpv(label, brick, hps=None):
    return HpVar(label, brick, hyperparameters=hps or {}, neighbor=HpInterval())
def const(label, cls):
    return Constant(label, cls, neighbor=ConstantInterval())
def cat(label, features, weights=None):
    return CatVar(label, features=features, weights=weights, neighbor=CatInterval())

cand_ops = [
    hpv("SelectFeatures", const("SelectFeaturesOp", SelectFeatures),
        {"feature_indices": cat("feature_indices", all_combos, weights=combo_weights)}),
    hpv("UnaryOp",  cat("UnaryOpType", [Identity, Inverse, Negate])),
    hpv("Power",    const("PowerOp", Power),
        {"exponent": cat("exponent", [-3, -2, -1, 1, 2, 3])}),
    hpv("Sum",      const("SumOp", SumFeatures)),
    hpv("Ln",       const("LnOp", Ln)),
    hpv("Sin",      const("SinOp", Sin)),
    hpv("Cos",      const("CosOp", Cos)),
    hpv("Exp",      const("ExpOp", Exp)),
    hpv("ExpAffine",const("ExpAffineOp", ExpAffine)),
    hpv("ChannelBoost", const("ChannelBoostOp", ChannelBoost),
        {"mode": cat("mode", ["add", "sub", "mul", "div"])}),
    hpv("ConstantBrick", const("ConstOp", ConstantBrick)),
]
operations = operations_var(
    "CandidateOperations", size=MAX_NODES, candidates=cand_ops,
    combiner_features=["add", "mul"],
    activations=Constant("id", value=nn.Identity(), neighbor=ConstantInterval()),
    node_type=SymbolicNode,
)
dag = EvoDagVariable(label="Dag", operations=operations, init_complexity=4,
                     neighbor=EvoDagInterval(nb_mutations=2))
search_space = ArrayVar(dag, label="Search Space", neighbor=ArrayInterval())
print(f"Search space: {len(search_space)} variables")

## 4. Dataset

In [ ]:
class MetaArchi(nn.Module):
    def __init__(self, args, input_shape):
        super().__init__()
        self.dag = args['Dag']
        self.dag.set(input_shape)
    def forward(self, X):
        return self.dag(X)


class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.y = torch.FloatTensor(y.values)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(RegressionDataset(X, y), batch_size=256, shuffle=False)


def forward(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            preds.append(model(Xb))
            trues.append(yb.reshape(-1, 1))
    return torch.cat(preds), torch.cat(trues)

## 5. Loss Function

In [ ]:
from dragon.utils.symbolic.loss_function.ols_pipeline import evaluate as ols_evaluate
from dragon.utils.symbolic.dag_to_formula import graph_to_all_formulas
from dragon.utils.symbolic.formula_extraction import (
    channel_formula, ols_formula,
)


def loss_function(args, idx):
    labels = [e.label for e in search_space]
    args_dict = ({labels[0]: args} if isinstance(args, AdjMatrix)
                 else dict(zip(labels, args)))

    model = MetaArchi(args_dict, input_shape=(num_features,))
    pred_all, true_all = forward(model, train_loader)

    loss, selected_c, ols_w, _, _, _, _, valid_idx, analysis = \
        ols_evaluate(pred_all, true_all, search_loss='corr', loss_mode='full')

    formulas = graph_to_all_formulas(
        model.dag.matrix, top_features, model.dag.operations, parse_sympy=False)

    f = ols_formula(formulas, analysis)
    if f is None:
        f = channel_formula(formulas, selected_c, pred_all,
                            true_all.squeeze().numpy(), 'corr') or 'N/A'

    if idx % 10 == 0:
        print(f"  [{idx}] loss={loss:.8f}  formula={str(f)[:80]}")

    return loss, model

## 6. Run the Search

In [ ]:
from dragon.search_algorithm.ssea import SteadyStateEA

algo = SteadyStateEA(
    search_space, n_iterations=50, population_size=10, selection_size=5,
    evaluation=loss_function, save_dir="save/multivariate",
)
min_loss = algo.run()
print(f"\nBest loss: {min_loss}")

## 7. Inspect Results

In [ ]:
from dragon.utils.plot_functions import load_archi
from dragon.utils.symbolic.dag_to_formula import graph_to_all_formulas
from dragon.utils.symbolic.formula_extraction import ols_formula, channel_formula

best = load_archi("save/multivariate/best_model/x.pkl")
if isinstance(best, list):
    best = best[0]
print(best)

pred, true = forward(best, train_loader)
_, selected_c, ols_w, _, _, _, _, valid_idx, analysis = \
    ols_evaluate(pred, true, search_loss='corr', loss_mode='full')
formulas = graph_to_all_formulas(
    best.matrix, top_features, best.operations, parse_sympy=False)
f = ols_formula(formulas, analysis)
if f is None:
    f = channel_formula(formulas, selected_c, pred,
                        true.squeeze().numpy(), 'corr') or 'N/A'
print(f"\nExtracted formula: {f}")
print(f"True formula:     y = x0^2 + 2*x1*x2 - x3/|x4|")